# AutoGluon Modeling (Competition Optimized)

This notebook trains AutoGluon models using a configuration strictly optimized for the competition's data structure and metric.

## Key Configuration
- **Data Source**: Switchable between Raw and Augmented features.
- **Leakage Prevention**: Uses `groups='survey_id'` with `num_bag_folds=3`. 
  - *Why 3?* There are exactly 3 surveys (100k, 200k, 300k). Validating on an entire survey requires 3 folds (Leave-One-Group-Out). Using more folds is impossible with distinct groups.
- **Metric**: Custom weighted combination of Consumption MAPE (10%) and Poverty Rate MAPE (90%).

In [1]:
%pip install -U uv
!uv pip install autogluon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 71.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.12 environment at: /usr
Resolved 240 packages in 6.72s                                       m⠋ Resolving dependencies...                                                     
Prepared 50 packages in 1.35s                                            
Uninstalled 11 packages in 307ms
Installed 51 packages in 329ms                              
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + autogluon==1.5.0
 + autogluon-common==1.5.0
 + autogluon-core==1.5.0
 + autogluon-features==1.5.0
 + autogluon-multimodal==1.5.0
 + autogluon-tabular==1.5.0
 + autogluon-timeseries==1.5.0
 + chronos-forecasting==2.2.2
 - click==8.3.1
 + click==8.2.1
 - cliff==4.13.0
 + cliff==4.12.0
 + colorful==0.5.8
 + coreforecast==0.0.16
 - datasets==4.4.1
 + datasets==4.0.0
 - dill==0.4.0
 + dill==0.3.8
 + distlib==0.4.0
 + einx==0.3.0
 + evaluate=

In [2]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer

## 1. Configuration & Data Loading
Set `USE_AUGMENTED_DATA` to `True` to load the features generated by the genetic search.

In [3]:
# === CONFIGURATION ===
USE_AUGMENTED_DATA = False  # Set to True to use generated features
AUGMENTED_PATH = None
RAW_PATH = "/kaggle/input/poverty/train_hh_features.csv"
GT_PATH = "/kaggle/input/poverty/train_hh_gt.csv"
# =====================

def load_data(use_augmented):
    gt_df = pd.read_csv(GT_PATH)
    
    if use_augmented:
        print(f"Loading Augmented Data from: {AUGMENTED_PATH}")
        features_df = pd.read_csv(AUGMENTED_PATH)
    else:
        print(f"Loading Raw Data from: {RAW_PATH}")
        features_df = pd.read_csv(RAW_PATH)
        
    # Data generated by feature search might already have the target/meta merged.
    # Check columns to decide merge strategy.
    
    # Essential columns we need from GT if not present
    needed_cols = ['cons_ppp17', 'survey_id']
    
    # If augmented data was saved with target/survey_id, features_df might have them.
    # If not, or if using raw, we merge.
    cols_to_merge = [c for c in needed_cols if c not in features_df.columns]
    
    if cols_to_merge:
        # Assuming hhid is present for merge
        if 'hhid' not in features_df.columns and 'hhid' in gt_df.columns:
            # If hhid missing in features (unlikely for raw), we might have index alignment assumption
            # But safer to assume hhid exists. 
            # If this fails for augmented, ensure augmented csv included hhid.
            raise ValueError("features_df missing 'hhid' for merge.")
            
        merged_df = pd.merge(features_df, gt_df[['hhid'] + cols_to_merge], on='hhid', how='left')
        return merged_df
    else:
        return features_df

train_data = load_data(USE_AUGMENTED_DATA)
print(f"Data Shape: {train_data.shape}")
print(f"Columns: {list(train_data.columns[:5])} ...")

Loading Raw Data from: /kaggle/input/poverty/train_hh_features.csv
Data Shape: (104234, 89)
Columns: ['hhid', 'com', 'weight', 'strata', 'utl_exp_ppp17'] ...


## 2. Define Custom Competition Metric for AutoGluon
We calculate the loss exactly as defined in the competition. Note that for AutoGluon, we typically define a metric where "higher is better" or specify `greater_is_better=False`. Here we use `greater_is_better=False` because it's a loss.

In [4]:
# Copy data (fine as-is)
%cp -r "/kaggle/input/poverty/comp_metric.py" "/kaggle/working/"

from autogluon.core.metrics import make_scorer
from comp_metric import CompetitionMetricExact

# Build metric object (CALLABLE, pickle-safe)
metric_obj = CompetitionMetricExact(
    survey_id_lookup=train_data["survey_id"]
)

# Fit thresholds ONCE (as intended by competition)
metric_obj.fit_thresholds_from_training(
    train_data["cons_ppp17"].values,
    train_data["survey_id"].values,
)

# Build AutoGluon scorer (NO wrapper, NO closures)
ag_scorer = make_scorer(
    name="competition_loss_exact",
    score_func=metric_obj,   # <- direct callable
    optimum=0,
    greater_is_better=False
)


## 3. Train AutoGluon Predictor

In [ ]:
label = 'cons_ppp17'
save_path = 'ag_models_competition'

predictor = TabularPredictor(
    label=label, 
    eval_metric=ag_scorer,
    path=save_path,
    groups='survey_id'  # Usage hint
)

# Explicitly verify groups column exists
assert 'survey_id' in train_data.columns, "survey_id missing!"

predictor.fit(
    train_data,
    presets='best_quality',
    # num_bag_folds=3,        # MANDATORY: Must be <= Number of Groups (3). Default (8) would crash.
    time_limit=3600*2       # 2 Hours time limit (adjust as needed)
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Sep 27 10:16:09 UTC 2025
CPU Count:          4
Pytorch Version:    2.8.0+cu126
CUDA Version:       CUDA is not available
Memory Avail:       29.77 GB / 31.35 GB (94.9%)
Disk Space Avail:   19.35 GB / 19.52 GB (99.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` v

(_ray_fit pid=833) <class 'numpy.ndarray'> <class 'numpy.ndarray'>


(_dystack pid=575) 	Warning: Exception caused LightGBMXT_BAG_L1 to fail during training... Skipping this model.
(_dystack pid=575) 		ray::_ray_fit() (pid=833, ip=172.19.2.2)
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/core/models/ensemble/fold_fitting_strategy.py", line 473, in _ray_fit
(_dystack pid=575)     fold_model.fit(X=X_fold, y=y_fold, X_val=X_val_fold, y_val=y_val_fold, time_limit=time_limit_fold, **resources, **kwargs_fold)
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/core/models/abstract/abstract_model.py", line 1125, in fit
(_dystack pid=575)     out = self._fit(**kwargs)
(_dystack pid=575)           ^^^^^^^^^^^^^^^^^^^
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/models/lgb/lgb_model.py", line 335, in _fit
(_dystack pid=575)     self.model = train_lgb_model(early_stopping_callback_kwargs=early_stopping_callback_kwargs, **train_params)
(_dystack pid=575)                 

(_ray_fit pid=969) <class 'numpy.ndarray'> <class 'numpy.ndarray'>


(_dystack pid=575) 	Warning: Exception caused LightGBM_BAG_L1 to fail during training... Skipping this model.
(_dystack pid=575) 		ray::_ray_fit() (pid=969, ip=172.19.2.2)
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/core/models/ensemble/fold_fitting_strategy.py", line 473, in _ray_fit
(_dystack pid=575)     fold_model.fit(X=X_fold, y=y_fold, X_val=X_val_fold, y_val=y_val_fold, time_limit=time_limit_fold, **resources, **kwargs_fold)
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/core/models/abstract/abstract_model.py", line 1125, in fit
(_dystack pid=575)     out = self._fit(**kwargs)
(_dystack pid=575)           ^^^^^^^^^^^^^^^^^^^
(_dystack pid=575)   File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/models/lgb/lgb_model.py", line 335, in _fit
(_dystack pid=575)     self.model = train_lgb_model(early_stopping_callback_kwargs=early_stopping_callback_kwargs, **train_params)
(_dystack pid=575)                  ^

(_ray_fit pid=1103) <class 'pandas.core.series.Series'> <class 'numpy.ndarray'>
(_ray_fit pid=1101) <class 'pandas.core.series.Series'> <class 'numpy.ndarray'>
(_ray_fit pid=1102) <class 'pandas.core.series.Series'> <class 'numpy.ndarray'>


(_dystack pid=575) 	-29.1109	 = Validation score   (-competition_loss_exact)
(_dystack pid=575) 	616.34s	 = Training   runtime
(_dystack pid=575) 	11.12s	 = Validation runtime
(_dystack pid=575) Fitting model: CatBoost_BAG_L1 ... Training model for up to 540.79s of the 1135.36s of remaining time.
(_dystack pid=575) 	Fitting 3 child models (S1F1 - S1F3) | Fitting with ParallelLocalFoldFittingStrategy (3 workers, per: cpus=1, gpus=0, memory=2.01%)


In [ ]:
# Leaderboard
predictor.leaderboard()